In [2]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np

# ================= 1. 可配置的隐私引擎 (逻辑不变) =================
class Comparable_LDP_Engine:
    def __init__(self, model, epsilon=50.0, strategy='adaptive'):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy  # 'uniform' or 'adaptive'
        self.layer_roles = self._identify_layers()

    def _identify_layers(self):
        roles = {}
        for name, _ in self.model.named_parameters():
            if any(f"model.{i}." in name for i in range(10)): roles[name] = "backbone"
            elif "Detect" in name or "head" in name: roles[name] = "head"
            else: roles[name] = "neck"
        return roles

    def step(self, current_epoch):
        if current_epoch < 3: return # Warmup

        current_device = next(self.model.parameters()).device
        sensitivities = []
        param_groups = []
        names_list = []

        for name, p in self.model.named_parameters():
            if p.requires_grad and p.grad is not None:
                sensitivities.append(p.grad.norm(2).item())
                param_groups.append(p)
                names_list.append(name)
        
        if not sensitivities: return

        # ============ 核心差异区域 ============
        factors = []
        if self.strategy == 'uniform':
            # 【对手策略】Standard DPSGD: 所有层权重一样 (1.0)
            factors = [1.0] * len(names_list)
        
        elif self.strategy == 'adaptive':
            # 【你的策略】SA-LDP: 骨干 0.5, 头部 2.0
            for n in names_list:
                role = self.layer_roles.get(n, "neck")
                if role == "backbone": factors.append(0.5) 
                elif role == "head":   factors.append(2.0) 
                else:                  factors.append(1.0)
        # ====================================

        factors = torch.tensor(factors, device=current_device)
        # 归一化，确保总预算一致 (公平对比)
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_groups:
            layer_eps = self.epsilon * weights[idx]
            clip_val = 10.0 
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            sigma = c * clip_val / (layer_eps + 1e-8)
            p.grad.add_(torch.randn_like(p.grad) * sigma)
            idx += 1

# ================= 2. 定义两个独立的训练器 (绕过参数检查) =================

# 🐢 训练器 A: 模拟传统方法 (Uniform)
class UniformTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # 强制指定为 uniform
        self.privacy_engine = Comparable_LDP_Engine(model, epsilon=50.0, strategy='uniform')
        return model

    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# 🐇 训练器 B: 你的方法 (Adaptive)
class AdaptiveTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # 强制指定为 adaptive
        self.privacy_engine = Comparable_LDP_Engine(model, epsilon=50.0, strategy='adaptive')
        return model

    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 8 对比实验 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

print("🚀 开始 Exp 8: 你的方法 (SA-LDP) vs 传统方法 (Standard LDP)...")

# --- Exp 8.1: Standard LDP (Uniform) ---
print("\n🐢 [Exp 8.1] Standard LDP (Baseline)...")
# 注意：这里不再传 'strategy' 参数
trainer_std = UniformTrainer(overrides={
    'model': 'yolo11n.pt',
    'data': FULL_YAML,
    'epochs': 40,
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '8.1_Standard_LDP',
    'device': '0',
    'exist_ok': True
})
trainer_std.train()

# --- Exp 8.2: SA-LDP (Ours) ---
print("\n🐇 [Exp 8.2] SA-LDP (Ours)...")
trainer_ours = AdaptiveTrainer(overrides={
    'model': 'yolo11n.pt',
    'data': FULL_YAML,
    'epochs': 40,
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '8.2_Ours_SA_LDP',
    'device': '0',
    'exist_ok': True
})
trainer_ours.train()

print("\n🏆 Exp 8 对比完成！请查看 mAP 差异。")

🚀 开始 Exp 8: 你的方法 (SA-LDP) vs 传统方法 (Standard LDP)...

🐢 [Exp 8.1] Standard LDP (Baseline)...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=8.1_Standard_LDP,

In [5]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np

# ================= 1. 带监控的隐私引擎 =================
class Comparable_LDP_Engine:
    def __init__(self, model, epsilon=10.0, strategy='adaptive'):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        self.layer_roles = self._identify_layers()

    def _identify_layers(self):
        roles = {}
        for name, _ in self.model.named_parameters():
            if any(f"model.{i}." in name for i in range(10)): roles[name] = "backbone"
            elif "Detect" in name or "head" in name: roles[name] = "head"
            else: roles[name] = "neck"
        return roles

    def step(self, current_epoch):
        # Warmup 前3轮不加噪
        if current_epoch < 3: return

        current_device = next(self.model.parameters()).device
        
        # 收集梯度
        param_groups = []
        names_list = []
        for name, p in self.model.named_parameters():
            if p.requires_grad and p.grad is not None:
                param_groups.append(p)
                names_list.append(name)
        
        if not param_groups: return

        # 策略选择
        factors = []
        if self.strategy == 'uniform':
            factors = [1.0] * len(names_list)
        elif self.strategy == 'adaptive':
            for n in names_list:
                role = self.layer_roles.get(n, "neck")
                if role == "backbone": factors.append(0.5) 
                elif role == "head":   factors.append(2.0) 
                else:                  factors.append(1.0)

        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors) 
        
        # 🔥【监控点】只在每个 Epoch 的第一个 batch 打印一次，防止刷屏
        # 我们用一个简单的 hack：检查 param_groups[0] 的梯度是否已经被修改过（很难判断），
        # 或者简单粗暴地：我们直接打印。为了不刷屏，你可以忽略，或者只看是不是有输出。
        # 这里为了验证，我们打印第一层的噪声系数
        if np.random.rand() < 0.001: # 随机抽样打印，证明它在运行
            print(f"⚡ [DEBUG] Epoch {current_epoch} | Strategy: {self.strategy} | Injecting Noise... (Epsilon={self.epsilon})")

        idx = 0
        for p in param_groups:
            layer_eps = self.epsilon * weights[idx]
            clip_val = 10.0 
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            sigma = c * clip_val / (layer_eps + 1e-8)
            p.grad.add_(torch.randn_like(p.grad) * sigma)
            idx += 1

# ================= 2. 训练器定义 =================
class UniformTrainerDebug(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # 🔵 Uniform, Eps=10
        self.privacy_engine = Comparable_LDP_Engine(model, epsilon=10.0, strategy='uniform')
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch) # 注入噪声
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

class AdaptiveTrainerDebug(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # 🔴 Adaptive, Eps=10
        self.privacy_engine = Comparable_LDP_Engine(model, epsilon=10.0, strategy='adaptive')
        return model

    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch) # 注入噪声
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 9 (Clean) =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

print("🚀 开始 Exp 9 (Clean Cache & Debug Mode)...")

print("\n🐢 [Exp 9.1] Standard LDP (High Noise E=10)...")
trainer_std = UniformTrainerDebug(overrides={
    'model': 'yolo11n.pt',
    'data': FULL_YAML,
    'epochs': 30, # 30轮够了
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '9.1_Standard_Eps10_Clean', # 改名防止复用
    'device': '0',
    'exist_ok': True
})
trainer_std.train()

print("\n🐇 [Exp 9.2] SA-LDP (High Noise E=10)...")
trainer_ours = AdaptiveTrainerDebug(overrides={
    'model': 'yolo11n.pt',
    'data': FULL_YAML,
    'epochs': 30,
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '9.2_Ours_Eps10_Clean', # 改名防止复用
    'device': '0',
    'exist_ok': True
})
trainer_ours.train()

🚀 开始 Exp 9 (Clean Cache & Debug Mode)...

🐢 [Exp 9.1] Standard LDP (High Noise E=10)...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=9.1_Standard_Eps10_Cl

In [4]:
import os
import shutil
import glob

# ================= 配置你的数据路径 =================
DATA_ROOT = "/root/autodl-tmp/exp1/NEU_DET_YOLO" # 请确认这是你的数据根目录

print(f"🧹 正在清理 {DATA_ROOT} 下的缓存文件...")

# 1. 删除 .cache 文件 (YOLO 的数据索引缓存)
cache_files = glob.glob(os.path.join(DATA_ROOT, "**/*.cache"), recursive=True)
for f in cache_files:
    try:
        os.remove(f)
        print(f"   - 已删除: {f}")
    except Exception as e:
        print(f"   ! 删除失败 {f}: {e}")

# 2. (可选) 清理之前的 Exp 8, Exp 9 文件夹，防止断点续传干扰
# 如果你不想手动删，可以在训练代码里把 name 改成 '9.1_New_Run' 这种
print("✅ 缓存清理完成！下次训练将重新索引数据。")

🧹 正在清理 /root/autodl-tmp/exp1/NEU_DET_YOLO 下的缓存文件...
   - 已删除: /root/autodl-tmp/exp1/NEU_DET_YOLO/labels/val.cache
   - 已删除: /root/autodl-tmp/exp1/NEU_DET_YOLO/labels/train.cache
✅ 缓存清理完成！下次训练将重新索引数据。


In [6]:
import os
from ultralytics import YOLO
import torch
import torch.nn as nn
import numpy as np

# ================= 1. 定义回调函数 (Callback) =================
# 这是 YOLO 最底层的注入方式，绝对管用！

def get_privacy_callback(epsilon=10.0, strategy='adaptive'):
    def privacy_callback(trainer):
        # -----------------------------------------------------------
        # 1. 策略定义区 (这里调整参数！)
        # -----------------------------------------------------------
        backbone_weight = 1.0
        head_weight = 1.0
        
        if strategy == 'adaptive':
            # 🔥 修正策略：温和一点，别把 Backbone 搞瞎了
            backbone_weight = 0.8  
            head_weight = 1.5      
        
        # -----------------------------------------------------------
        # 2. 识别层级
        # -----------------------------------------------------------
        model = trainer.model
        current_device = next(model.parameters()).device
        
        param_groups = []
        names_list = []
        for name, p in model.named_parameters():
            if p.requires_grad and p.grad is not None:
                param_groups.append(p)
                names_list.append(name)
        
        if not param_groups: return

        # -----------------------------------------------------------
        # 3. 计算噪声权重
        # -----------------------------------------------------------
        factors = []
        for n in names_list:
            if strategy == 'uniform':
                factors.append(1.0)
            else:
                # 简单粗暴的层级判断
                if any(f"model.{i}." in n for i in range(10)): 
                    factors.append(backbone_weight)
                elif "Detect" in n or "head" in n: 
                    factors.append(head_weight)
                else: 
                    factors.append(1.0) # Neck 部分保持默认

        factors = torch.tensor(factors, device=current_device)
        # 归一化：保证总预算公平！
        weights = factors / factors.sum() * len(factors) 

        # -----------------------------------------------------------
        # 4. 注入噪声 (In-place)
        # -----------------------------------------------------------
        # 监控：每个 epoch 的第一个 batch 打印一次
        if trainer.batch_idx == 0:
            print(f"⚡ [DEBUG] Epoch {trainer.epoch} | Strategy: {strategy} | Backbone W: {backbone_weight} | Noise Injecting...")

        idx = 0
        for p in param_groups:
            layer_eps = epsilon * weights[idx]
            # 梯度裁剪 (Clip)
            clip_val = 10.0 
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            # 加噪
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            sigma = c * clip_val / (layer_eps + 1e-8)
            p.grad.add_(torch.randn_like(p.grad) * sigma)
            idx += 1
            
    return privacy_callback

# ================= 2. 执行实验 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

print("🚀 开始 Exp 9_Final: 终极修正版 (Epsilon=10.0)...")

# --- Exp 9.1: Standard LDP (Baseline) ---
print("\n🐢 [Exp 9.1] Standard LDP (Uniform)...")
model_std = YOLO('yolo11n.pt')
# 注册回调：on_before_optimizer_step (在优化器更新前执行，完美时机)
model_std.add_callback("on_before_optimizer_step", get_privacy_callback(epsilon=10.0, strategy='uniform'))

model_std.train(
    data=FULL_YAML,
    epochs=30,
    batch=16,
    imgsz=640,
    project='result_exp1',
    name='9.1_Standard_Final',
    device='0',
    exist_ok=True
)

# --- Exp 9.2: SA-LDP (Ours) ---
print("\n🐇 [Exp 9.2] SA-LDP (Ours - Tuned)...")
model_ours = YOLO('yolo11n.pt')
# 注册回调：使用 adaptive 策略
model_ours.add_callback("on_before_optimizer_step", get_privacy_callback(epsilon=10.0, strategy='adaptive'))

model_ours.train(
    data=FULL_YAML,
    epochs=30,
    batch=16,
    imgsz=640,
    project='result_exp1',
    name='9.2_Ours_Final',
    device='0',
    exist_ok=True
)

🚀 开始 Exp 9_Final: 终极修正版 (Epsilon=10.0)...

🐢 [Exp 9.1] Standard LDP (Uniform)...
New https://pypi.org/project/ultralytics/8.3.249 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=tra

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f266c101750>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [7]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np

# ================= 1. 隐私引擎 (含新参数) =================
class PrivacyEngine_Final:
    def __init__(self, model, epsilon=10.0, strategy='uniform'):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        self.layer_roles = self._identify_layers()

    def _identify_layers(self):
        roles = {}
        for name, _ in self.model.named_parameters():
            if any(f"model.{i}." in name for i in range(10)): roles[name] = "backbone"
            elif "Detect" in name or "head" in name: roles[name] = "head"
            else: roles[name] = "neck"
        return roles

    def step(self, current_epoch):
        if current_epoch < 3: return

        current_device = next(self.model.parameters()).device
        
        # 收集梯度
        param_groups = []
        names_list = []
        for name, p in self.model.named_parameters():
            if p.requires_grad and p.grad is not None:
                param_groups.append(p)
                names_list.append(name)
        
        if not param_groups: return

        # 🔥 策略配置区 🔥
        factors = []
        
        if self.strategy == 'uniform':
            # 对手：全 1.0
            factors = [1.0] * len(names_list)
            
        elif self.strategy == 'adaptive_tuned':
            # 我们 (调优后)：Backbone 0.8, Head 1.5
            for n in names_list:
                role = self.layer_roles.get(n, "neck")
                if role == "backbone": factors.append(0.8)  # ⬅️ 改温和了 (原来是0.5)
                elif role == "head":   factors.append(1.5)  # ⬅️ 重点保护 (原来是2.0)
                else:                  factors.append(1.0)

        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors) 
        
        # 打印监控 (只打印第一个参数的梯度被修改，证明运行了)
        # 为了避免刷屏，我们用一个 flag
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"⚡ [DEBUG] Epoch {current_epoch} | Strategy: {self.strategy} | Injecting Noise... (Backbone Factor: {factors[0]:.2f})")
            self._logged_this_epoch = current_epoch

        idx = 0
        for p in param_groups:
            layer_eps = self.epsilon * weights[idx]
            clip_val = 10.0 
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            sigma = c * clip_val / (layer_eps + 1e-8)
            p.grad.add_(torch.randn_like(p.grad) * sigma)
            idx += 1

# ================= 2. 训练器 (继承法，最稳) =================

# 对手训练器
class Trainer_Uniform(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        self.privacy_engine = PrivacyEngine_Final(model, epsilon=10.0, strategy='uniform')
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch) # 👈 强制执行
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# 你的训练器
class Trainer_Adaptive_Tuned(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # 使用调优后的策略
        self.privacy_engine = PrivacyEngine_Final(model, epsilon=10.0, strategy='adaptive_tuned')
        return model

    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch) # 👈 强制执行
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 10 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

print("🚀 开始 Exp 10: 真正决胜局 (Epsilon=10.0, Tuned Params)...")

# --- 10.1: Standard ---
print("\n🐢 [Exp 10.1] Standard LDP (E=10)...")
trainer_std = Trainer_Uniform(overrides={
    'model': 'yolo11n.pt',
    'data': FULL_YAML,
    'epochs': 30,
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '10.1_Standard_E10',
    'device': '0',
    'exist_ok': True
})
trainer_std.train()

# --- 10.2: Ours (Tuned) ---
print("\n🐇 [Exp 10.2] SA-LDP Tuned (E=10)...")
trainer_ours = Trainer_Adaptive_Tuned(overrides={
    'model': 'yolo11n.pt',
    'data': FULL_YAML,
    'epochs': 30,
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '10.2_Ours_E10_Tuned',
    'device': '0',
    'exist_ok': True
})
trainer_ours.train()

🚀 开始 Exp 10: 真正决胜局 (Epsilon=10.0, Tuned Params)...

🐢 [Exp 10.1] Standard LDP (E=10)...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=10.1_Standard_E10, nb

In [9]:
import torch
from ultralytics import YOLO
import sys

def verify_and_fix_strategy(model_path='yolo11n.pt'):
    print(f"\n🔍 正在进行深度层级扫描...")
    
    model = YOLO(model_path)
    py_model = model.model
    
    # ================= 1. 动态寻找 Head 层的索引 =================
    head_module_indices = []
    
    print(f"{'层级索引':<10} | {'模块类型'}")
    print("-" * 40)
    
    # 遍历所有子模块，找到属于 Detect 类的模块索引
    for name, module in py_model.named_modules():
        # name 类似 "model.0", "model.23"
        if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
            print(f"🎯 找到 Head 模块: {name} ({module.__class__.__name__})")
            # 提取数字索引，例如 "model.23" -> "23"
            try:
                idx = name.split('.')[1]
                head_module_indices.append(idx)
            except:
                pass
    
    if not head_module_indices:
        print("❌ 警告：未在模型中找到 'Detect' 模块！")
        return
    
    print(f"✅ Head 层索引确认为: {head_module_indices}")
    print("-" * 40)

    # ================= 2. 模拟参数分配逻辑 (修复版) =================
    
    # 打印前我们需要看到所有参数（不管是否 requires_grad）
    param_list = list(py_model.named_parameters())
    print(f"📊 模型参数总数: {len(param_list)}")
    
    matched_backbone = 0
    matched_head = 0
    matched_neck = 0
    
    print(f"\n{'参数名 (抽样)':<40} | {'原始判定'} | {'修正后判定'} | {'权重'}")
    print("-" * 80)
    
    for i, (name, p) in enumerate(param_list):
        # 修正后的识别逻辑
        role = "neck"
        weight = 1.0 # Standard
        
        # 1. 识别 Backbone (前10层，通常是 model.0 到 model.9)
        # 简单粗暴检查字符串 'model.X.'
        is_backbone = False
        for k in range(10):
            if f"model.{k}." in name:
                is_backbone = True
                break
        
        # 2. 识别 Head (使用刚才动态找到的索引)
        is_head = False
        for idx in head_module_indices:
            if f"model.{idx}." in name:
                is_head = True
                break
        
        # 3. 分配角色
        if is_backbone:
            role = "backbone"
            weight = 0.8
            matched_backbone += 1
        elif is_head:
            role = "head"
            weight = 1.5 # 重点保护
            matched_head += 1
        else:
            role = "neck"
            weight = 1.0
            matched_neck += 1
            
        # 抽样打印 (打印 Backbone开头, Neck开头, Head开头)
        if i < 2 or (i > len(param_list)//2 and i < len(param_list)//2 + 2) or is_head:
             # 为了避免 Head 刷屏，只打印 Head 的前两个
            if is_head:
                 # 简单计数器 hack
                if not hasattr(verify_and_fix_strategy, f"head_count_{idx}"):
                     setattr(verify_and_fix_strategy, f"head_count_{idx}", 0)
                
                cnt = getattr(verify_and_fix_strategy, f"head_count_{idx}")
                if cnt < 2:
                    print(f"{name:<40} | {'Unknown'} | {role:<10} | {weight}")
                    setattr(verify_and_fix_strategy, f"head_count_{idx}", cnt + 1)
            else:
                print(f"{name:<40} | {'Unknown'} | {role:<10} | {weight}")

    print("-" * 80)
    print(f"📈 统计结果:")
    print(f"   - Backbone 参数组数: {matched_backbone}")
    print(f"   - Neck     参数组数: {matched_neck}")
    print(f"   - Head     参数组数: {matched_head}")
    
    if matched_head > 0:
        print("\n🎉 修正成功！现在代码可以精准定位到 Head 层了！")
    else:
        print("\n❌ 修正失败，依然没有匹配到 Head 参数。")

# 运行验证
verify_and_fix_strategy()


🔍 正在进行深度层级扫描...
层级索引       | 模块类型
----------------------------------------
🎯 找到 Head 模块:  (DetectionModel)
🎯 找到 Head 模块: model.23 (Detect)
✅ Head 层索引确认为: ['23']
----------------------------------------
📊 模型参数总数: 256

参数名 (抽样)                                 | 原始判定 | 修正后判定 | 权重
--------------------------------------------------------------------------------
model.0.conv.weight                      | Unknown | backbone   | 0.8
model.0.bn.weight                        | Unknown | backbone   | 0.8
model.13.m.0.cv2.conv.weight             | Unknown | neck       | 1.0
model.23.cv2.0.0.conv.weight             | Unknown | head       | 1.5
model.23.cv2.0.0.bn.weight               | Unknown | head       | 1.5
--------------------------------------------------------------------------------
📈 统计结果:
   - Backbone 参数组数: 99
   - Neck     参数组数: 90
   - Head     参数组数: 67

🎉 修正成功！现在代码可以精准定位到 Head 层了！


In [10]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np

# ================= 1. 智能隐私引擎 (Smart Privacy Engine) =================
class PrivacyEngine_Smart:
    def __init__(self, model, epsilon=10.0, strategy='uniform'):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        # 🔥 关键：初始化时自动扫描结构
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10)) # YOLO通常0-9是backbone

    def _find_head_indices(self):
        # 动态扫描模型结构，寻找 Detect 模块
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    # name 可能是 "model.23"
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                    print(f"🎯 [Engine] 自动锁定检测头层级: model.{idx}")
                except:
                    pass
        if not indices:
             # 如果找不到，兜底使用 23 (YOLOv11n 默认)
            print("⚠️ [Engine] 未自动找到 Head，使用默认索引 [23]")
            indices = [23]
        return indices

    def step(self, current_epoch):
        # Warmup
        if current_epoch < 3: return

        current_device = next(self.model.parameters()).device
        
        param_groups = []
        names_list = []
        for name, p in self.model.named_parameters():
            if p.requires_grad and p.grad is not None:
                param_groups.append(p)
                names_list.append(name)
        
        if not param_groups: return

        # 🔥 策略配置 🔥
        factors = []
        
        if self.strategy == 'uniform':
            factors = [1.0] * len(names_list)
            
        elif self.strategy == 'adaptive_smart':
            for n in names_list:
                # 解析层级索引 (例如 model.0.conv -> 0)
                try:
                    # 假设名字格式为 "model.X.xxx"
                    layer_idx = int(n.split('.')[1])
                except:
                    layer_idx = -1 # 解析失败归为 neck
                
                # 判定角色
                if layer_idx in self.head_indices:
                    factors.append(1.5) # 🛡️ Head: 重点保护 (噪声小)
                elif layer_idx in self.backbone_indices:
                    factors.append(0.8) # 📉 Backbone: 适度牺牲 (噪声大)
                else:
                    factors.append(1.0) # Neck: 标准

        factors = torch.tensor(factors, device=current_device)
        # 归一化
        weights = factors / factors.sum() * len(factors) 
        
        # 调试打印 (只打印第一个 Head 参数的权重，证明策略生效)
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            # 找一个 head 参数看看权重
            head_w = -1
            for i, n in enumerate(names_list):
                 # 简单判断是否包含 head 索引
                 for h_idx in self.head_indices:
                     if f"model.{h_idx}." in n:
                         head_w = weights[i].item()
                         break
                 if head_w != -1: break
            
            print(f"⚡ [DEBUG] Epoch {current_epoch} | {self.strategy} | Head Weight: {head_w:.4f} (Target: >1.0)")
            self._logged_this_epoch = current_epoch

        idx = 0
        for p in param_groups:
            layer_eps = self.epsilon * weights[idx]
            clip_val = 10.0 
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            sigma = c * clip_val / (layer_eps + 1e-8)
            p.grad.add_(torch.randn_like(p.grad) * sigma)
            idx += 1

# ================= 2. 训练器定义 =================

class Trainer_Standard(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        self.privacy_engine = PrivacyEngine_Smart(model, epsilon=10.0, strategy='uniform')
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

class Trainer_Smart(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # 使用智能策略
        self.privacy_engine = PrivacyEngine_Smart(model, epsilon=10.0, strategy='adaptive_smart')
        return model

    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 10_Real_Final =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

print("🚀 开始 Exp 10_Real_Final: 真正的智能隐私保护 (E=10)...")

# --- 10.1: Standard (Baseline) ---
print("\n🐢 [Exp 10.1] Standard LDP (Uniform)...")
trainer_std = Trainer_Standard(overrides={
    'model': 'yolo11n.pt',
    'data': FULL_YAML,
    'epochs': 30,
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '10.1_Standard_Fixed',
    'device': '0',
    'exist_ok': True
})
trainer_std.train()

# --- 10.2: Ours (Smart) ---
print("\n🐇 [Exp 10.2] SA-LDP (Smart Structure-Aware)...")
trainer_smart = Trainer_Smart(overrides={
    'model': 'yolo11n.pt',
    'data': FULL_YAML,
    'epochs': 30,
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '10.2_Ours_Smart',
    'device': '0',
    'exist_ok': True
})
trainer_smart.train()

🚀 开始 Exp 10_Real_Final: 真正的智能隐私保护 (E=10)...

🐢 [Exp 10.1] Standard LDP (Uniform)...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=10.1_Standard_Fixed, nbs=

In [11]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np

# ================= 1. 包含 AGC 的智能隐私引擎 =================
class PrivacyEngine_Advanced:
    def __init__(self, model, epsilon=10.0, strategy='uniform', adaptive_clip=False):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        self.adaptive_clip = adaptive_clip # 🔥 新增开关
        
        # 自动寻找 Head
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except:
                    pass
        if not indices: indices = [23]
        return indices

    def step(self, current_epoch):
        if current_epoch < 3: return # Warmup

        current_device = next(self.model.parameters()).device
        
        param_groups = []
        names_list = []
        grad_norms = [] # 用于存储所有梯度的范数

        for name, p in self.model.named_parameters():
            if p.requires_grad and p.grad is not None:
                param_groups.append(p)
                names_list.append(name)
                # 计算该参数的梯度范数
                grad_norms.append(p.grad.norm(2).item())
        
        if not param_groups: return

        # 🔥【核心改进】自适应裁剪阈值 (AGC) 🔥
        if self.adaptive_clip:
            # 取本次迭代中所有梯度范数的中位数
            # 这样 Clip 值会随着训练进行自动变小！
            current_median = np.median(grad_norms)
            # 限制 C 在 [0.1, 5.0] 之间，防止过大或过小
            clip_val = max(0.1, min(current_median * 1.5, 10.0)) 
        else:
            clip_val = 10.0 # 笨办法：固定值

        # 策略分配 (权重计算)
        factors = []
        if self.strategy == 'uniform':
            factors = [1.0] * len(names_list)
        elif self.strategy == 'adaptive_smart':
            for n in names_list:
                try:
                    layer_idx = int(n.split('.')[1])
                except:
                    layer_idx = -1
                
                if layer_idx in self.head_indices:
                    factors.append(1.5) # Head 保护
                elif layer_idx in self.backbone_indices:
                    factors.append(0.8) # Backbone 牺牲
                else:
                    factors.append(1.0)

        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors) 
        
        # 调试打印 (检查 Clip 值的变化)
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            if self.adaptive_clip:
                print(f"⚡ [AGC DEBUG] Epoch {current_epoch} | Adaptive Clip C: {clip_val:.4f} (Dynamic!)")
            self._logged_this_epoch = current_epoch

        idx = 0
        for p in param_groups:
            layer_eps = self.epsilon * weights[idx]
            
            # 1. 使用动态的 clip_val 进行裁剪
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            # 2. 噪声的大小 sigma 与 clip_val 成正比
            # 因为 clip_val 变小了，所以加的噪声 sigma 也自动变小了！
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            sigma = c * clip_val / (layer_eps + 1e-8)
            
            p.grad.add_(torch.randn_like(p.grad) * sigma)
            idx += 1

# ================= 2. 训练器定义 =================

# 之前的标准版 (Baseline)
class Trainer_Standard(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        self.privacy_engine = PrivacyEngine_Advanced(model, epsilon=10.0, strategy='uniform', adaptive_clip=False)
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# 🔥 你的加强版 (SA-LDP + AGC)
class Trainer_Advanced(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # 开启 adaptive_clip=True
        self.privacy_engine = PrivacyEngine_Advanced(model, epsilon=10.0, strategy='adaptive_smart', adaptive_clip=True)
        return model

    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 11 (决死一战) =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

print("🚀 开始 Exp 11: SA-LDP + 自适应梯度裁剪 (AGC)...")
print("这次我们不仅优化了空间(Layer)，还优化了时间(Epoch)！")

# 11.1 Standard (之前跑过了，可以注释掉，或者重跑做对比)
# print("\n🐢 [Exp 11.1] Standard (Baseline)...")
# trainer_std = Trainer_Standard(overrides={'model':'yolo11n.pt', 'data':FULL_YAML, 'epochs':30, 'batch':16, 'project':'Thesis_Exp', 'name':'11.1_Std', 'exist_ok':True})
# trainer_std.train()

# 11.2 Advanced (Ours + AGC)
print("\n🔥 [Exp 11.2] SA-LDP + AGC (The Ultimate)...")
trainer_adv = Trainer_Advanced(overrides={
    'model': 'yolo11n.pt',
    'data': FULL_YAML,
    'epochs': 30,
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '11.2_Ours_AGC',
    'device': '0',
    'exist_ok': True
})
trainer_adv.train()

🚀 开始 Exp 11: SA-LDP + 自适应梯度裁剪 (AGC)...
这次我们不仅优化了空间(Layer)，还优化了时间(Epoch)！

🔥 [Exp 11.2] SA-LDP + AGC (The Ultimate)...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=Fal

In [12]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np

# ================= 1. 引擎定义 (保持 Exp 11 的最强配置) =================
# ================= 1. 替换这个引擎类 (只修改了 step 函数) =================
class PrivacyEngine_Ultimate:
    def __init__(self, model, epsilon=10.0, strategy='adaptive_smart', adaptive_clip=True):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        self.adaptive_clip = adaptive_clip
        
        # 自动寻找 Head 索引
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except:
                    pass
        if not indices: indices = [23]
        return indices

    def step(self, current_epoch):
        # Warmup 前3轮不操作
        if current_epoch < 3: return

        # 🔥【新增功能】只在第3轮刚开始时，打印一次全网“安检报告”
        if not hasattr(self, '_has_printed_structure'):
            print(f"\n📋 [Exp 12 终极安检] Epoch {current_epoch} 网络状态快照:")
            print(f"{'层级名称 (抽样)':<40} | {'状态':<10} | {'隐私策略'}")
            print("-" * 70)
            
            # 遍历所有参数（包括冻结的）
            for i, (name, p) in enumerate(self.model.named_parameters()):
                # 只抽样打印：前几层(Backbone)、中间(Neck)、最后几层(Head)
                if i < 3 or (i > 100 and i < 103) or "Detect" in name or "model.23" in name:
                    
                    # 1. 判断是否冻结
                    status = "❄️ 冻结" if not p.requires_grad else "🔥 训练"
                    
                    # 2. 判断权重
                    weight_info = "无噪声"
                    if p.requires_grad:
                        try:
                            layer_idx = int(name.split('.')[1])
                        except:
                            layer_idx = -1
                        
                        if layer_idx in self.head_indices:
                            weight_info = "🛡️ 保护 (1.5)"
                        elif layer_idx in self.backbone_indices:
                            weight_info = "⚠️ 牺牲 (0.8)"
                        else:
                            weight_info = "⚡ 标准 (1.0)"
                    
                    print(f"{name:<40} | {status:<10} | {weight_info}")
            
            print("-" * 70)
            self._has_printed_structure = True  # 锁住，后面不再打印

        # ================= 以下是原本的加噪逻辑 (保持不变) =================
        current_device = next(self.model.parameters()).device
        param_groups = []
        names_list = []
        grad_norms = []

        # 这一步只收集“需要梯度”的参数，冻结层会自动被这里忽略，不会加噪
        for name, p in self.model.named_parameters():
            if p.requires_grad and p.grad is not None:
                param_groups.append(p)
                names_list.append(name)
                grad_norms.append(p.grad.norm(2).item())
        
        if not param_groups: return

        # AGC
        if self.adaptive_clip:
            current_median = np.median(grad_norms) if grad_norms else 1.0
            clip_val = max(0.1, min(current_median * 1.5, 10.0)) 
        else:
            clip_val = 10.0

        # 计算权重
        factors = []
        for n in names_list:
            try:
                layer_idx = int(n.split('.')[1])
            except:
                layer_idx = -1
            
            if layer_idx in self.head_indices:
                factors.append(1.5)
            elif layer_idx in self.backbone_indices:
                factors.append(0.8)
            else:
                factors.append(1.0)

        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_groups:
            layer_eps = self.epsilon * weights[idx]
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            sigma = c * clip_val / (layer_eps + 1e-8)
            p.grad.add_(torch.randn_like(p.grad) * sigma)
            idx += 1

# ================= 2. 训练器定义 =================
class Trainer_Ultimate(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        # 注意：这里我们加载预训练权重，而不是 yolo11n.pt
        model = super().get_model(cfg, weights, verbose)
        self.privacy_engine = PrivacyEngine_Ultimate(model, epsilon=10.0, strategy='adaptive_smart', adaptive_clip=True)
        return model

    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 12 (终极一战) =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
# 🔥 关键：使用 Exp 2.3 的预训练权重 (请确认路径存在)
PRETRAINED_WEIGHTS = "/root/autodl-tmp/exp1/result_exp1/2.3_DS_Pretrain/weights/best.pt"

print("🚀 开始 Exp 12: 终极融合 (Transfer + SA-LDP + AGC)...")

if os.path.exists(PRETRAINED_WEIGHTS):
    trainer_ultimate = Trainer_Ultimate(overrides={
        'model': PRETRAINED_WEIGHTS, # 👈 换成最强底座
        'data': FULL_YAML,
        'epochs': 30,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_exp1',
        'name': '12_Ultimate_Fusion',
        'device': '0',
        'exist_ok': True,
        'freeze': 10 # ❄️ 冻结前10层！保护迁移过来的强壮特征不被噪声破坏
    })
    trainer_ultimate.train()
else:
    print(f"❌ 找不到预训练权重: {PRETRAINED_WEIGHTS}")
    print("请检查 Exp 2.3 是否运行成功。")

🚀 开始 Exp 12: 终极融合 (Transfer + SA-LDP + AGC)...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/2.3_DS_Pretrain/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=12_Ultima

In [2]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np

# ================= 1. Fisher 增强版隐私引擎 (修复设备报错版) =================
class PrivacyEngine_Fisher:
    def __init__(self, model, epsilon=10.0, strategy='adaptive_smart', beta=0.9):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        self.beta = beta
        
        # 1. 初始化梯度历史缓存
        # 我们只建立字典结构，具体 Tensor 的设备在 step 中动态对齐
        self.grad_history = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                # 先暂时放在 p 当前的设备上 (可能是 CPU)
                self.grad_history[name] = torch.zeros_like(p.data)

        # 自动寻找 Head
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def step(self, current_epoch):
        current_device = next(self.model.parameters()).device
        
        # Warmup 前3轮不加噪，但要积累 Fisher 信息
        if current_epoch < 3: 
            for name, p in self.model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    # 🔥 修复核心：设备动态对齐 🔥
                    if self.grad_history[name].device != p.device:
                        self.grad_history[name] = self.grad_history[name].to(p.device)
                    
                    new_grad_sq = p.grad.data.pow(2)
                    self.grad_history[name].mul_(self.beta).add_(new_grad_sq * (1 - self.beta))
            return

        # ================= 正式加噪流程 =================
        
        # 1. 收集梯度范数 (AGC 用)
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        if not grad_norms: return

        # 2. AGC 计算阈值
        current_median = np.median(grad_norms)
        clip_val = max(0.1, min(current_median * 1.5, 10.0))

        # 调试打印
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"⚡ [Fisher DEBUG] Epoch {current_epoch} | AGC Clip: {clip_val:.4f} | Device: {current_device}")
            self._logged_this_epoch = current_epoch

        # 3. 逐层处理
        for name, p in self.model.named_parameters():
            if not p.requires_grad or p.grad is None: continue

            # 🔥 修复核心：确保历史缓存也在 GPU 上 🔥
            if self.grad_history[name].device != p.device:
                self.grad_history[name] = self.grad_history[name].to(p.device)

            # --- A. 更新 Fisher ---
            new_grad_sq = p.grad.data.pow(2)
            self.grad_history[name].mul_(self.beta).add_(new_grad_sq * (1 - self.beta))

            # --- B. 计算微观权重 (Fisher Scaler) ---
            importance = self.grad_history[name].sqrt()
            
            # 稳定性处理
            avg_imp = importance.mean()
            importance = torch.clamp(importance, min=avg_imp * 0.1) 
            
            noise_scaler = 1.0 / (importance + 1e-8)
            # 归一化：保证层级总噪声能量守恒
            noise_scaler.div_(noise_scaler.mean() + 1e-8)
            # 截断极端值
            noise_scaler.clamp_(0.1, 5.0)

            # --- C. 计算宏观权重 (Layer Factor) ---
            layer_factor = 1.0
            try:
                layer_idx = int(name.split('.')[1])
                if self.strategy == 'adaptive_smart':
                    if layer_idx in self.head_indices: layer_factor = 1.5
                    elif layer_idx in self.backbone_indices: layer_factor = 0.8
            except: pass

            # --- D. 加噪 ---
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (self.epsilon * layer_factor + 1e-8)
            
            raw_noise = torch.randn_like(p.grad) * base_sigma
            final_noise = raw_noise * noise_scaler # Fisher 加权
            
            p.grad.add_(final_noise)

# ================= 2. 训练器 =================
class Trainer_Fisher(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # 启用 Fisher
        self.privacy_engine = PrivacyEngine_Fisher(model, epsilon=10.0, strategy='adaptive_smart')
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 15 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

print("🚀 开始 Exp 15: Fisher 参数级自适应隐私保护...")
print("融合三要素: SA-LDP (空间) + AGC (时间) + Fisher (参数)")

trainer_fisher = Trainer_Fisher(overrides={
    'model': 'yolo11n.pt', # 先用通用底座验证有效性
    'data': FULL_YAML,
    'epochs': 30,
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '15_Fisher_Opt',
    'device': '0',
    'exist_ok': True
})
trainer_fisher.train()

🚀 开始 Exp 15: Fisher 参数级自适应隐私保护...
融合三要素: SA-LDP (空间) + AGC (时间) + Fisher (参数)
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=15_Fisher_Opt, nbs=64, nms=Fals

KeyboardInterrupt: 

In [3]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np

# ================= 1. Fisher 安全版隐私引擎 (User's Safe Logic + Device Fix) =================
class PrivacyEngine_Fisher_Safe:
    def __init__(self, model, epsilon=10.0, strategy='adaptive_smart', beta=0.9):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        self.beta = beta
        
        # 1. 初始化梯度历史 (暂时在 CPU/当前设备)
        self.grad_history = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.grad_history[name] = torch.zeros_like(p.data)

        # Head 索引
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def step(self, current_epoch):
        # 🛡️ 机制 3: 延迟介入 (Late Start)
        # 前 5 轮只做基础加噪，不开启 Fisher 缩放，让模型先收敛一下
        use_fisher = (current_epoch >= 5)

        # -----------------------------------------------------------
        # 🔥 关键修复：动态设备对齐 (防止 RuntimeError)
        # -----------------------------------------------------------
        current_device = next(self.model.parameters()).device
        
        # 如果是 Warmup 阶段或者 Late Start 前，也要更新历史，但要注意设备
        if not use_fisher:
            for name, p in self.model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    # 搬运 history 到 GPU
                    if self.grad_history[name].device != p.device:
                        self.grad_history[name] = self.grad_history[name].to(p.device)
                    
                    new_grad_sq = p.grad.data.pow(2)
                    self.grad_history[name].mul_(self.beta).add_(new_grad_sq * (1 - self.beta))
        # -----------------------------------------------------------

        # 收集梯度范数
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        if not grad_norms: return

        # === AGC 计算 (安全版) ===
        current_median = np.median(grad_norms)
        # 下限设为 0.01，防止后期梯度太小时被强行放大
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        # 调试打印
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"🛡️ [Fisher-Safe] Epoch {current_epoch} | AGC: {clip_val:.4f} | Fisher: {use_fisher} | Device: {current_device}")
            self._logged_this_epoch = current_epoch

        # === 遍历参数处理 ===
        for name, p in self.model.named_parameters():
            if not p.requires_grad or p.grad is None: continue

            # 🔥 再次确保设备对齐
            if self.grad_history[name].device != p.device:
                self.grad_history[name] = self.grad_history[name].to(p.device)

            # --- A. 更新参数重要性历史 ---
            new_grad_sq = p.grad.data.pow(2)
            self.grad_history[name].mul_(self.beta).add_(new_grad_sq * (1 - self.beta))

            # --- B. 计算 Fisher 权重 ---
            noise_scaler = None
            if use_fisher:
                # 🛡️ 机制 1: 分母加厚 (Robust Epsilon 1e-4)
                importance = self.grad_history[name].sqrt().add(1e-4)
                
                raw_scaler = 1.0 / importance
                
                # 归一化
                avg_scaler = raw_scaler.mean()
                raw_scaler.div_(avg_scaler + 1e-8)
                
                # 🛡️ 机制 2: 绝对截断 (Hard Clamping [0.5, 3.0])
                # 这是防止 NaN 的核心！
                noise_scaler = raw_scaler.clamp(min=0.5, max=3.0)

            # --- C. 层级系数 (SA-LDP) ---
            layer_factor = 1.0
            try:
                layer_idx = int(name.split('.')[1])
                if self.strategy == 'adaptive_smart':
                    if layer_idx in self.head_indices: layer_factor = 1.5
                    elif layer_idx in self.backbone_indices: layer_factor = 0.8
            except: pass

            # --- D. 加噪 ---
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (self.epsilon * layer_factor + 1e-8)
            
            noise = torch.randn_like(p.grad) * base_sigma
            
            # 应用 Fisher
            if noise_scaler is not None:
                noise.mul_(noise_scaler)
            
            p.grad.add_(noise)

# ================= 2. 训练器定义 =================
class Trainer_Fisher_Safe(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        self.privacy_engine = PrivacyEngine_Fisher_Safe(model, epsilon=10.0, strategy='adaptive_smart')
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 15_Retry =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

print("🚀 开始 Exp 15_Retry: Fisher Safe Mode...")
print("✅ 已应用: Hard Clamp [0.5, 3.0], Late Start (Epoch 5), Robust Epsilon")

trainer_safe = Trainer_Fisher_Safe(overrides={
    'model': 'yolo11n.pt',
    'data': FULL_YAML,
    'epochs': 30,
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '15_Fisher_Safe_Retry',
    'device': '0',
    'exist_ok': True
})
trainer_safe.train()

🚀 开始 Exp 15_Retry: Fisher Safe Mode...
✅ 已应用: Hard Clamp [0.5, 3.0], Late Start (Epoch 5), Robust Epsilon
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=15_

KeyboardInterrupt: 

In [4]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np
import math

# ================= 1. 包含噪声退火的隐私引擎 =================
class PrivacyEngine_Annealing:
    def __init__(self, model, epsilon=10.0, strategy='adaptive_smart', epochs=30):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        self.total_epochs = epochs
        
        # 自动寻找 Head
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def _get_noise_multiplier(self, current_epoch):
        """
        计算当前的噪声衰减系数。
        使用余弦退火策略：从 1.0 平滑下降到 0.2
        """
        # 设定最小噪声比例 (防止完全没有隐私)
        min_decay = 0.2 
        
        # 余弦函数：0 -> 1.0,  max_epoch -> 0.2
        progress = current_epoch / self.total_epochs
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        
        # 映射到 [min_decay, 1.0]
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        if current_epoch < 3: return # Warmup

        # 1. 计算当前的退火系数
        decay_factor = self._get_noise_multiplier(current_epoch)

        # 2. AGC (自适应梯度裁剪)
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        if not grad_norms: return

        # 限制下限为 0.01
        current_median = np.median(grad_norms)
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        # 调试打印
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"📉 [Annealing DEBUG] Epoch {current_epoch} | AGC: {clip_val:.4f} | Noise Decay: {decay_factor:.4f}")
            self._logged_this_epoch = current_epoch

        # 3. 策略参数
        current_device = next(self.model.parameters()).device
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad]
        factors = []
        
        for n in names_list:
            layer_factor = 1.0
            try:
                layer_idx = int(n.split('.')[1])
                if self.strategy == 'adaptive_smart':
                    if layer_idx in self.head_indices: layer_factor = 1.5
                    elif layer_idx in self.backbone_indices: layer_factor = 0.8
            except: pass
            factors.append(layer_factor)

        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors) 
        
        # 4. 加噪主循环
        idx = 0
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                # 原始 epsilon 计算
                layer_eps = self.epsilon * weights[idx]
                torch.nn.utils.clip_grad_norm_(p, clip_val)
                
                c = np.sqrt(2 * np.log(1.25 / 1e-5))
                # 基础 Sigma
                base_sigma = c * clip_val / (layer_eps + 1e-8)
                
                # 🔥 核心改进：应用退火系数 🔥
                # 随着 Epoch 增加，Sigma 逐渐变小
                final_sigma = base_sigma * decay_factor
                
                p.grad.add_(torch.randn_like(p.grad) * final_sigma)
                idx += 1

# ================= 2. 训练器 =================
class Trainer_Annealing(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # 传入总 Epochs 以便计算退火
        self.privacy_engine = PrivacyEngine_Annealing(model, epsilon=10.0, strategy='adaptive_smart', epochs=30)
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 16 (噪声退火) =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

# 依然建议使用 Transfer 的底座，如果不喜欢 Transfer，可以用 'yolo11n.pt'
# 这里为了验证算法纯粹性，我们先用 yolo11n.pt (Pure Algorithm Check)
# 如果你想刷分，就把下面的 model 换成 Exp 13 的 best.pt

print("🚀 开始 Exp 16: 噪声退火策略 (Noise Annealing)...")
print("✅ 解决后期不收敛问题，且数值绝对稳定！")

trainer_anneal = Trainer_Annealing(overrides={
    'model': 'yolo11n.pt',
    'data': FULL_YAML,
    'epochs': 30,
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '16_Noise_Annealing',
    'device': '0',
    'exist_ok': True
})
trainer_anneal.train()

🚀 开始 Exp 16: 噪声退火策略 (Noise Annealing)...
✅ 解决后期不收敛问题，且数值绝对稳定！
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=16_Noise_Annealing, nbs=64, nms=False, opset=No

In [5]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np

# ================= 1. 混合噪声隐私引擎 =================
class PrivacyEngine_Hybrid:
    def __init__(self, model, epsilon=10.0, strategy='adaptive_smart'):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        
        # 头部索引查找
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def step(self, current_epoch):
        # Warmup
        if current_epoch < 3: return

        # 1. 计算 AGC 阈值
        # 注意：为了统一标准，我们还是先用 L2 范数来确定“梯度的量级”
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        
        if not grad_norms: return
        
        current_median = np.median(grad_norms)
        # 裁剪阈值
        clip_val = max(0.1, min(current_median * 1.5, 10.0))

        # 调试打印
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"🧬 [Hybrid DEBUG] Epoch {current_epoch} | AGC: {clip_val:.4f} | Strategy: Head(Laplace)+Back(Gaussian)")
            self._logged_this_epoch = current_epoch

        # 2. 遍历参数应用混合策略
        for name, p in self.model.named_parameters():
            if not p.requires_grad or p.grad is None:
                continue
            
            # 判断层级
            layer_type = 'backbone'
            try:
                layer_idx = int(name.split('.')[1])
                if layer_idx in self.head_indices:
                    layer_type = 'head'
            except: pass

            # === 分支 A: Head 层 (使用 Laplace + L1 裁剪) ===
            if layer_type == 'head':
                # 1. L1 裁剪 (Manhattan Clipping)
                # Head 层我们希望它稀疏一点
                torch.nn.utils.clip_grad_norm_(p, clip_val, norm_type=1.0)
                
                # 2. 计算 Laplace 噪声尺度 b
                # Head 享受 1.5 倍权重 (即 epsilon * 1.5)
                layer_eps = self.epsilon * 1.5
                scale = clip_val / (layer_eps + 1e-8)
                
                # 3. 生成 Laplace 噪声
                # PyTorch 的 Laplace 分布采样
                m = torch.distributions.laplace.Laplace(loc=0.0, scale=scale)
                # 确保噪声在同一设备
                noise = m.sample(p.grad.shape).to(p.device)
                
                p.grad.add_(noise)

            # === 分支 B: Backbone 层 (使用 Gaussian + L2 裁剪) ===
            else:
                # 1. L2 裁剪 (Euclidean Clipping) - 保持稳健
                torch.nn.utils.clip_grad_norm_(p, clip_val, norm_type=2.0)
                
                # 2. 计算 Gaussian 噪声尺度 sigma
                # Backbone 权重 0.8
                layer_eps = self.epsilon * 0.8
                c = np.sqrt(2 * np.log(1.25 / 1e-5))
                sigma = c * clip_val / (layer_eps + 1e-8)
                
                # 3. 生成 Gaussian 噪声
                noise = torch.randn_like(p.grad) * sigma
                
                p.grad.add_(noise)

# ================= 2. 训练器 =================
class Trainer_Hybrid(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        self.privacy_engine = PrivacyEngine_Hybrid(model, epsilon=10.0, strategy='adaptive_smart')
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        # 这里把外部的全局裁剪去掉了，因为我们在 engine 内部做了更细致的 per-parameter 裁剪
        # torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0) 
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 17 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

print("🚀 开始 Exp 17: 混合噪声机制 (Hybrid Laplace-Gaussian)...")
print("🧪 实验假设: Laplace 噪声能让 Detection Head 的特征更稀疏、决策更果断。")

# 依然建议使用 Transfer 的底座，但为了控制变量，先用 yolo11n.pt
trainer_hybrid = Trainer_Hybrid(overrides={
    'model': 'yolo11n.pt', 
    'data': FULL_YAML,
    'epochs': 30,
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '17_Hybrid_Noise',
    'device': '0',
    'exist_ok': True
})
trainer_hybrid.train()

🚀 开始 Exp 17: 混合噪声机制 (Hybrid Laplace-Gaussian)...
🧪 实验假设: Laplace 噪声能让 Detection Head 的特征更稀疏、决策更果断。
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=17_Hybrid_